[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BSLJunhyeonJeon/AI_COP/blob/main/session2/notebooks/03_labeling_sam2.ipynb)

# session2 · 03 · 실습2 · SAM2 라벨링 체험 (혈액세포)

- **이 노트북에서 배우는 것**: SAM2로 **클릭(포인트) 한 번에 세포 마스크**가 생기는 걸 체험한다. (분할 라벨을 이렇게 만든다)
- **입력**: 없음 — 혈액 도말 1장을 코드로 확보(1단계 데이터 재사용).
- **출력**: "클릭점 → 마스크" 그림(`outputs/`) + 생성 마스크(`data/segmentation/`)

> **핵심**: 2단계의 일반 분할 모델(SegFormer)은 혈액에서 실패했지만, 파운데이션 모델 **SAM2는 학습 없이(zero-shot) 클릭만으로 세포를 잘 따낸다.** 단 **클래스(이름)는 사람이 지정**한다(SAM2는 '어디'만, '무엇'은 사람이).
> 학습/파인튜닝 없음(프롬프트 추론만). tiny SAM2 + GPU 있으면 사용·없으면 CPU. SAM2가 안 뜨면 Otsu 백업으로 흐름 유지.
> 진짜 마우스 클릭 대신 **좌표 점**으로 체험한다(코랩 위젯 불안정). 좌표를 바꿔 재실행하면 다른 세포를 '클릭'하는 셈.

In [ ]:
# 셀 1 · 환경 감지 + 프로젝트 루트 확보 (00/01/02 와 동일 패턴 — 분기는 이 셀 한 곳)
import os, subprocess

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

SESSION = "session2"
REPO_URL = "https://github.com/BSLJunhyeonJeon/AI_COP"
REPO_DIR = "/content/AI_COP"
SESSION_DIR = REPO_DIR + "/" + SESSION


def acquire_project():
    """코랩: 레포를 /content/AI_COP 로 클론 후 session2 를 PROJECT_ROOT 로 반환(실패 시 None)."""
    if os.path.isdir(REPO_DIR):
        print("이미 존재:", REPO_DIR, "(재클론 건너뜀)")
        try:
            r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
            if r.returncode != 0:
                print("  (git pull 실패 — 기존 캐시 버전 사용)")
        except Exception as e:
            print("  (git pull 건너뜀:", e, ")")
    else:
        print("레포 클론:", REPO_URL, "->", REPO_DIR)
        try:
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
        except Exception as e:
            print("clone 실패(네트워크/권한 확인):", e)
    return SESSION_DIR if os.path.isdir(SESSION_DIR) else None


def find_root_local(marker="requirements.txt"):
    """로컬: requirements.txt 를 마커로 상위 탐색해 session2 루트를 찾는다."""
    start = os.path.abspath(os.getcwd())
    d = start
    while True:
        if os.path.exists(os.path.join(d, marker)):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            print("[주의] '" + marker + "' 를 못 찾음. 현재 폴더를 루트로 가정:", start)
            return start
        d = parent


PROJECT_ROOT = acquire_project() if IN_COLAB else find_root_local()
if not (PROJECT_ROOT and os.path.isdir(PROJECT_ROOT)):
    raise RuntimeError(
        "세션 루트를 확보하지 못했습니다. "
        "코랩이면 레포 클론 실패이니 네트워크 확인 후 이 셀(셀 1)을 다시 ▶ 실행하세요. "
        "로컬이면 session2/ 안에서 노트북을 열었는지 확인하세요."
    )
os.chdir(PROJECT_ROOT)
import torch
print("실행 환경   :", "Colab" if IN_COLAB else "Local")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("장치(device):", "GPU(cuda)" if torch.cuda.is_available() else "CPU (느리지만 가능)")

In [ ]:
# 셀 2 · 의존성 확인 (SAM2용 새 deps 없음 — 2단계의 ultralytics 재사용, 버전만 확인)
import os, sys, subprocess, re

if not os.path.exists("requirements.txt"):
    raise RuntimeError("requirements.txt 를 찾지 못했습니다. 셀 1을 먼저 ▶ 실행하세요.")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
if r.returncode != 0:
    raise RuntimeError("pip install 실패. 위 로그 확인 후 이 셀(셀 2)을 다시 ▶ 실행하세요.")

import ultralytics
v = ultralytics.__version__
print("ultralytics:", v)
ver = tuple(int(x) for x in re.findall(r"\d+", v)[:3])
if ver < (8, 2, 70):
    print("[주의] SAM2 에는 ultralytics>=8.2.70 이 필요합니다. (현재", v, ") — requirements 업그레이드가 필요할 수 있습니다.")
else:
    print("-> SAM2 사용 가능 (ultralytics>=8.2.70).")

In [ ]:
# 셀 3 · 라벨링할 이미지 확보 (혈액 도말 1장 — 세포가 여러 개라 라벨링 체험에 적합)
import os, glob, subprocess
from PIL import Image

demo = os.path.join("data", "_demo_inputs")
os.makedirs(demo, exist_ok=True)
LABEL = os.path.join(demo, "label_smear.jpg")

if not os.path.exists(LABEL):
    ex = sorted(glob.glob(os.path.join("data", "detection", "*.jpg")))
    if ex:
        Image.open(ex[0]).convert("RGB").save(LABEL)
        print("도말 이미지(1단계 data/detection 재사용):", ex[0])
    else:
        src = os.path.join("data", "_bccd_src")
        if not os.path.isdir(src):
            subprocess.run(["git", "clone", "--depth", "1",
                            "https://github.com/Shenggan/BCCD_Dataset", src], check=False)
        bccd = sorted(glob.glob(os.path.join(src, "BCCD", "JPEGImages", "*.jpg")))
        if bccd:
            Image.open(bccd[0]).convert("RGB").save(LABEL)
            print("도말 이미지(BCCD 확보)")
        else:
            print("[주의] 도말 이미지 확보 실패(네트워크). 셀 3을 다시 ▶ 실행하세요.")

print("라벨링 이미지 준비:", os.path.exists(LABEL), "(", LABEL, ")")

In [ ]:
# 셀 4 · SAM2 클릭(포인트) -> 마스크 (이 노트북의 핵심)
# 점 2~3개로 "클릭마다 세포 하나가 잡힌다"를 보여준다. SAM2 실패 시 Otsu 백업으로 흐름 유지.
import os, numpy as np
import matplotlib.pyplot as plt
from PIL import Image

img_path = os.path.join("data", "_demo_inputs", "label_smear.jpg")
if not os.path.exists(img_path):
    raise RuntimeError("라벨링할 이미지가 없습니다. 셀 3을 먼저 ▶ 실행하세요.")
img = Image.open(img_path).convert("RGB")
W, H = img.size
# 디바이스는 ultralytics 가 자동 선택한다(GPU 있으면 사용). SAM 호출에 device 인자를 넣지 않는다.

# 예시 '클릭' 좌표(이미지 비율 기반). 좌표를 바꿔 재실행하면 다른 세포를 클릭하는 셈.
points = [[int(W * 0.40), int(H * 0.45)],
          [int(W * 0.62), int(H * 0.32)],
          [int(W * 0.30), int(H * 0.66)]]


def otsu_threshold(gray):
    hist = np.histogram(gray, bins=256, range=(0, 256))[0].astype(float)
    total = gray.size
    sum_all = np.dot(np.arange(256), hist)
    sumB, wB, best, thr = 0.0, 0.0, -1.0, 0
    for t in range(256):
        wB += hist[t]
        if wB == 0:
            continue
        wF = total - wB
        if wF == 0:
            break
        sumB += t * hist[t]
        mB, mF = sumB / wB, (sum_all - sumB) / wF
        v = wB * wF * (mB - mF) ** 2
        if v > best:
            best, thr = v, t
    return thr


def to_hw_bool(m):
    """마스크를 원본 (H,W) bool 로 맞춘다."""
    m = np.asarray(m).astype(np.uint8)
    if m.shape != (H, W):
        m = np.array(Image.fromarray(m * 255).resize((W, H))) > 127
    return m.astype(bool)


masks, method = [], None
try:
    from ultralytics import SAM
    sam, last_err = None, None
    for name in ["sam2_t.pt", "sam2.1_t.pt"]:          # tiny SAM2 변형(앞에서부터 시도, 자동 다운로드)
        try:
            sam = SAM(name)
            method = "SAM2 (%s)" % name
            break
        except Exception as e:
            last_err = e
            print("  (SAM2 로드 실패:", name, "->", e, ")")
    if sam is None:
        raise RuntimeError("SAM2 모델 로드 실패: %s" % last_err)
    for pt in points:
        # 점 1개 = 객체 1개 (스펙 확인 API: points=[[x,y]], labels=[1])
        res = sam(img_path, points=[pt], labels=[1], verbose=False)[0]
        if res.masks is not None and len(res.masks.data) > 0:
            masks.append((pt, to_hw_bool(res.masks.data[0].cpu().numpy())))
    if not masks:
        raise RuntimeError("마스크가 비어 있음")
    print("SAM2 마스크 생성:", len(masks), "개 (클릭 점마다 1개)  모델:", method)
except Exception as e:
    print("[주의] SAM2 로드/실행 실패:", e)
    print("-> 백업(Otsu)으로 흐름을 시연합니다(라이브에서 SAM2가 안 떠도 수업이 안 깨지게).")
    print("   참고: 백업은 '클릭별'이 아니라 전체 세포 영역을 한 번에 잡는 단일 임시 마스크입니다.")
    method = "Otsu(backup)"
    gray = np.array(img.convert("L"))
    masks = [(points[0], to_hw_bool((gray < otsu_threshold(gray)).astype(np.uint8)))]

# 저장 (data/segmentation/ — .gitignore 로 비커밋)
seg = os.path.join("data", "segmentation")
os.makedirs(seg, exist_ok=True)
for i, (pt, m) in enumerate(masks):
    Image.fromarray((m * 255).astype(np.uint8)).save(os.path.join(seg, "clickmask_%d.png" % i))

# 시각화: (왼) 클릭점  (오) 마스크 오버레이
colors = [(255, 0, 0), (0, 255, 0), (0, 128, 255), (255, 255, 0)]
ov = np.array(img).copy()
for i, (pt, m) in enumerate(masks):
    ov[m] = (0.5 * ov[m] + 0.5 * np.array(colors[i % len(colors)])).astype(np.uint8)
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(img)
for pt in points:
    ax[0].plot(pt[0], pt[1], "r*", markersize=15)
ax[0].set_title("Clicks (points)"); ax[0].axis("off")
ax[1].imshow(ov)
ax[1].set_title("%s masks (one click -> one cell)" % method); ax[1].axis("off")
fig.suptitle("One click (point) -> one cell mask  (zero-shot; class assigned by human)")
os.makedirs("outputs", exist_ok=True)
plt.tight_layout()
plt.savefig(os.path.join("outputs", "03_sam2_clicks.png"), dpi=130, bbox_inches="tight")
plt.show()
print("저장: outputs/03_sam2_clicks.png, data/segmentation/clickmask_*.png  (방식:", method, ")")

In [ ]:
# 셀 5 · 라벨링 워크플로 + 대비
# 클릭 = 마스크(SAM2가 '어디')  +  클래스 = 사람이 지정('무엇')  ->  라벨 1개 완성
import os, glob, numpy as np
import matplotlib.pyplot as plt
from PIL import Image

img_path = os.path.join("data", "_demo_inputs", "label_smear.jpg")
mask_paths = sorted(glob.glob(os.path.join("data", "segmentation", "clickmask_*.png")))
human_classes = ["WBC", "RBC", "RBC", "Platelet"]   # 사람이 붙이는 예시 클래스(SAM2는 '무엇'인지 모름)

if os.path.exists(img_path) and mask_paths:
    base = np.array(Image.open(img_path).convert("RGB"))
    ov = base.copy()
    colors = [(255, 0, 0), (0, 255, 0), (0, 128, 255), (255, 255, 0)]
    fig, ax = plt.subplots(figsize=(7, 6))
    for i, mp in enumerate(mask_paths):
        m = np.array(Image.open(mp).convert("L").resize((base.shape[1], base.shape[0]))) > 127
        ov[m] = (0.5 * ov[m] + 0.5 * np.array(colors[i % len(colors)])).astype(np.uint8)
        ys, xs = np.where(m)
        if len(xs):
            ax.text(xs.mean(), ys.mean(), human_classes[i % len(human_classes)], color="white",
                    fontsize=11, ha="center", bbox=dict(facecolor="black", alpha=0.5, pad=1))
    ax.imshow(ov)
    ax.set_title("Click = mask (SAM2)  +  Class = human-assigned  ->  one label")
    ax.axis("off")
    os.makedirs("outputs", exist_ok=True)
    plt.tight_layout()
    plt.savefig(os.path.join("outputs", "03_labeling_workflow.png"), dpi=130, bbox_inches="tight")
    plt.show()
    print("저장: outputs/03_labeling_workflow.png")
else:
    print("[주의] 마스크가 없습니다. 셀 4를 먼저 ▶ 실행하세요.")

print("워크플로: 손으로 폴리곤을 일일이 그리는 대신 '클릭 1번 -> 마스크'. 클래스 이름만 사람이 붙이면 라벨 완성.")
print("체험 팁 : 셀 4의 points 좌표를 바꿔 다시 실행하면 다른 세포를 '클릭'하는 셈입니다.")
print("참고    : 실제 라벨링 도구로는 CVAT + SAM 통합이 있습니다(커리큘럼).")

In [ ]:
# 셀 6 · 종합 + 다음 세션으로의 다리 (검증 포함)
import os, glob

masks = glob.glob(os.path.join("data", "segmentation", "clickmask_*.png"))
figs = [p for p in ["outputs/03_sam2_clicks.png", "outputs/03_labeling_workflow.png"] if os.path.exists(p)]

print("=" * 56)
print(" 3단계 · SAM2 라벨링 체험 검증/요약")
print("=" * 56)
print(" - 생성 마스크 파일 :", len(masks), "개  (data/segmentation/clickmask_*.png)")
print(" - 체험 그림        :", len(figs), "개  (", ", ".join(figs) if figs else "없음", ")")
if not figs:
    print(" [주의] 그림이 없습니다 — 셀 3·4·5 를 순서대로 다시 ▶ 실행하세요.")
print("=" * 56)
print("정리:")
print(" SAM2는 학습 없이(zero-shot) '클릭(어디)'만으로 세포 마스크를 따낸다.")
print(" '무엇(클래스)'은 사람이 붙인다 -> 클릭 + 클래스 = 분할 라벨 1개.")
print(" => 이렇게 만든 라벨로 다음 세션에 모델을 전이학습한다.")